# Лекция: Кластерный анализ в Python

**Дисциплина:** Введение в анализ больших данных

**Кластеризация** — разбиение объектов на группы так, чтобы внутри группы объекты были похожи, а между группами — различались.

Методы:
1. **Иерархические** — дендрограмма, `linkage` + `fcluster`
2. **Нейерархические** — k-means (число кластеров задаётся заранее)

Перед кластеризацией признаки обычно **стандартизируют**.

Демо: **penguins** (морфометрия). Примеры **не из лабораторного задания** — его выполните самостоятельно.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Данные и стандартизация


In [ ]:
peng = sns.load_dataset("penguins").dropna()
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = peng[features].copy()
print(X.describe().round(1))


In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features,
    index=X.index,
)
print(X_scaled.head().round(3))
print("средние ≈ 0:", X_scaled.mean().round(3).tolist())
print("ст.откл. ≈ 1:", X_scaled.std(ddof=0).round(3).tolist())


---
## 2. Иерархическая кластеризация

`linkage(X, method="ward")` → дендрограмма → `fcluster(..., criterion="maxclust")`.


In [ ]:
idx = X_scaled.sample(40, random_state=1).index
Xs = X_scaled.loc[idx]
Z = linkage(Xs, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, labels=Xs.index.astype(str).tolist(), leaf_rotation=90, leaf_font_size=8)
plt.title("Дендрограмма (Ward), n=40")
plt.ylabel("Расстояние")
plt.tight_layout()
plt.show()


In [ ]:
Z_full = linkage(X_scaled, method="ward")
labels_h = fcluster(Z_full, t=3, criterion="maxclust")
peng_h = peng.copy()
peng_h["cluster_h"] = labels_h
print("Размеры кластеров:")
print(peng_h["cluster_h"].value_counts().sort_index())
print("\nСредние признаков по кластерам:")
print(peng_h.groupby("cluster_h")[features].mean().round(1))


---
## 3. K-means

Число кластеров: **elbow** (inertia) и **silhouette**.


In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    lab = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lab))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_range), inertias, "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow")
axes[1].plot(list(K_range), silhouettes, "o-")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title("Silhouette")
plt.tight_layout()
plt.show()


In [ ]:
k_opt = 3
km = KMeans(n_clusters=k_opt, n_init=25, random_state=42)
labels_km = km.fit_predict(X_scaled)

peng_km = peng.copy()
peng_km["cluster"] = labels_km + 1
print("Размеры:", peng_km["cluster"].value_counts().sort_index().to_dict())
print("\nЦентры (в стандартизированных координатах):")
print(pd.DataFrame(km.cluster_centers_, columns=features).round(3))
print("\nСредние в исходных единицах:")
print(peng_km.groupby("cluster")[features].mean().round(1))


In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=labels_km, cmap="tab10",
                 s=40, edgecolors="k", alpha=0.8)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"K-means, k={k_opt} (PCA 2D)")
plt.colorbar(sc, label="cluster")
plt.tight_layout()
plt.show()
print("Доля дисперсии PC1+PC2:", round(pca.explained_variance_ratio_.sum(), 3))


### Сравнение с известными видами (только для иллюстрации)

В реальном задании «истинных» меток нет — здесь смотрим, насколько кластеры совпадают с `species`.


In [ ]:
ct = pd.crosstab(peng_km["species"], peng_km["cluster"])
print(ct)
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")
plt.title("species × cluster (k-means)")
plt.tight_layout()
plt.show()


### Как описать результаты

1. Обоснуйте число кластеров (дендрограмма / elbow / silhouette).  
2. Опишите **профили** кластеров (средние признаков).  
3. При необходимости визуализируйте в PCA-плоскости.

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Стандартизация | `StandardScaler().fit_transform(X)` |
| Иерархия | `linkage(X, method="ward")` |
| Дендрограмма | `dendrogram(Z, labels=...)` |
| Срез на k групп | `fcluster(Z, t=k, criterion="maxclust")` |
| K-means | `KMeans(n_clusters=k, n_init=10).fit(X)` |
| Метки / центры | `km.labels_`, `km.cluster_centers_` |
| Inertia | `km.inertia_` |
| Silhouette | `silhouette_score(X, labels)` |

---
## Что сделать после лекции

1. Повторите иерархию и k-means на **других** числовых столбцах.  
2. Откройте лабораторное задание и выполните кластеризацию **самостоятельно**.  
3. Всегда стандартизируйте признаки перед расчётом расстояний.

Удачи!
